# The Flip the Script Pattern - Forcing Clarification and Exploration

Hello everyone. In nearly every pattern so far, the dynamic has been the same: we give a task, and the AI provides an answer. But what if the task is ambiguous? What if we want the model to make fewer assumptions about the task it is about to complete?

The **\"Flip the Script\" Pattern** changes this dynamic. Instead of getting an answer, our goal is to get **intelligent questions or a set of well-defined options** from the AI. This turns the model from a passive answering machine into a proactive, Socratic partner, helping us refine our thinking before committing to a solution.

This notebook will demonstrate the two primary use cases for this pattern:

1. Forcing the AI to ask clarifying questions for an ambiguous task.
2. Asking the AI to explore a solution space and present options.

## Helper Functions

First, let's set up our standard helper functions. The \"Flip the Script\" pattern is entirely about how we craft the prompt, so our standard functions will work perfectly.

In [ ]:
import litellm
from IPython.display import display, Markdown
from textwrap import dedent
from dotenv import load_dotenv

load_dotenv()

MODEL_NAME = "openai/gpt-4o-mini"
MAX_TOKENS_DEFAULT = 500

def get_completion(
    prompt,
    model=MODEL_NAME,
    max_tokens=MAX_TOKENS_DEFAULT,
    **kwargs
):
    if "gpt-5" in model:
        kwargs["max_completion_tokens"] = max_tokens
    else:
        kwargs["max_tokens"] = max_tokens
        
    parsed_messages = []

    if type(prompt) is str:
        parsed_messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]
    else:
        parsed_messages = prompt

    response = litellm.completion(
        model=model,
        messages=parsed_messages,
        **kwargs
    )

    return response.choices[0].message.content

print("Setup complete. Helper functions and code context are ready.")

## Use Case 1: Forcing Clarification on Ambiguous Tasks

As developers, we often start with a vague requirement like \"build an auth system.\" A bad prompt would ask the AI to just generate the code, forcing it to make dozens of incorrect assumptions.

A good prompt \"flips the script\" and uses the AI's knowledge to help us define the problem better.

In [ ]:
prompt_clarification = [
    {
        "role": "system",
        "content": "You are a senior software architect and expert consultant."
    },
    {
        "role": "user",
        "content": dedent("""
        Please design a user authentication system for a web application.

        Before providing any code or specific solutions, please ask as many questions as you need
        to clarify assumptions, implementation guidelines and project characteristics.
        """)
    }
]

result_clarification = get_completion(prompt_clarification)

In [ ]:
display(Markdown(result_clarification))

## Use Case 2: Exploring an Open-Ended Solution Space

Sometimes the problem isn't ambiguous, but the number of possible solutions is overwhelming. We can \"flip the script\" to turn the AI into a consultant that lays out our options and lets us choose the path forward.

In [ ]:
prompt_exploration = [
    {
        "role": "system",
        "content": "You are an expert database architect."
    },
    {
        "role": "user",
        "content": dedent("""
        I need to choose a database for a new social media application.

        Your task is to provide a list of 3 suitable database options:
        1. A traditional SQL option.
        2. A NoSQL document DB option.
        3. A NoSQL graph DB option.

        For each option, provide a short summary of its primary strength for the use case.
        After listing and explaining the options, ask me how I want to proceed.
        """)
    }
]

result_exploration = get_completion(prompt_exploration)

In [ ]:
display(Markdown(result_exploration))